# Assignment 3 — Code Submission (Jupyter Notebook Template)

**Submit:** this `.ipynb` file inside the required ZIP package, together with the required wav_examples/ folder.

**Group members (Name | ID):**  
- `<Yizhe Xu> | <a1989279>`  
- `<Zhiyang Ma> | <a1975426>`  
- `<NAME 3> | <ID 3>`  
- `<NAME 4> | <ID 4>`

**Date:** `<DD Month YYYY>`

---

## What the marker should be able to do (end-to-end)
From a fresh environment, the marker should be able to:

1. **Install dependencies** (from your provided dependencies file)  
2. **Run training** for each experiment (or run your provided trained checkpoints, if allowed by the task sheet)  
3. **Run inference**: input plain text → output a playable `.wav` file  
4. **See evidence**: objective metric(s) + qualitative samples (≥10 text prompts with corresponding `.wav` outputs)

---

## Expected artefacts / recommended structure
You can adapt this, but keep it **clear and reproducible**.

- `dependencies.*` (choose one): `requirements.txt` / `environment.yml` / `pyproject.toml`  
- `scripts/`
  - `train.py` (or `train_expA.py`, `train_expB.py`, etc.)
  - `synth.py` (single “text → wav” entrypoint recommended)
  - `evaluate.py` (objective metric computation)
- `configs/` (optional but recommended)
  - `expA.yaml`, `expB.yaml`, ...
- `results/`
  - `wavs/` (generated audio samples; **≥10 prompts**)
  - `metrics.json` or `metrics.csv`
  - `spectrograms/` (optional)

> Note: Spectrograms are **optional** unless your task sheet explicitly requires them. If you include them, provide 2–3 representative examples per experiment.

---

## How to use this notebook
- Use the headings below as your “evidence checklist”.
- Add **code cells** directly under each heading for implementation and outputs (tables, logs, plots, audio).
- Keep paths consistent and relative where possible.

## B integration status and execution notes

The original dataset implementation and its saved outputs from A are retained below.
B adds portable path configuration, a shared training program, and a checkpoint reload helper.
The new B cells have not yet been run on the group's GPU. Their empty outputs do not represent results.
Before submission, restart the selected project kernel and rerun the full notebook.

Run this notebook from the repository root. Install `requirements.txt` using the CUDA-aware
command in `README_B.md`. Model and dataset files are downloaded separately and are not submitted.


In [12]:
# Shared local paths. Select your verified Python as the notebook kernel.
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
local_data = Path(r"E:\work\datasets\LJSpeech-1.1")
default_data = str(local_data) if local_data.is_dir() else "LJSpeech-1.1"
os.environ.setdefault("LJSPEECH_ROOT", default_data)
local_cache = Path(r"E:\work\_cache\huggingface")
default_cache = str(local_cache) if os.name == "nt" and Path("E:/work").is_dir() else str(Path.home() / ".cache/huggingface")
os.environ.setdefault("TTS_HF_CACHE", default_cache)
os.environ["HF_HOME"] = os.environ["TTS_HF_CACHE"]
print("Python:", sys.executable)
print("LJSpeech:", os.environ["LJSPEECH_ROOT"])
print("Model cache:", os.environ["HF_HOME"])


Python: e:\work\TTS_A3\.venv\Scripts\python.exe
LJSpeech: E:\work\datasets\LJSpeech-1.1
Model cache: E:\work\_cache\huggingface


# 1.1 Dataset handling & pre-processing (Rubric 1.1)

## 1.1.1 Dataset location and structure

- **Dataset:** LJSpeech
- **Local path:** `LJSpeech-1.1/`
- **Metadata:** `metadata.csv`
- **Audio directory:** `wavs/`

Each metadata entry contains an utterance ID, the original text, and the normalised text. We retain both text fields and use the normalised text as the starting point for model preprocessing.

We check for malformed metadata rows, duplicate or empty IDs, empty normalised texts, and missing audio files.

Dataset source: https://keithito.com/LJ-Speech-Dataset/

## 1.1.2 Audio preprocessing (document your choices)

- **Sampling rate:** Inspect all original WAV headers. Resample audio to 16,000 Hz on loading to meet SpeechT5 requirements, using `scipy.signal.resample_poly` with anti-aliasing filtering.
- **Channels and format:** Convert loaded audio to a one-dimensional mono float32 waveform.
- **Silence trimming / padding:** No silence trimming or manual padding is applied during loading. Batch padding will be handled by the training data collator.
- **Filtering / normalisation:** No additional denoising or amplitude normalisation is applied.
- **Sanity checks:** Report original sample rates, channel counts, and duration statistics. Check three processed examples for valid values, format, and preserved duration, and provide paired text and audio for listening.
- **Original files:** Preserve all original WAV files; apply the same loading function across experiments.

## 1.1.3 Text preprocessing

- **Text source:** Use the normalised LJSpeech transcription as the starting point.
- **Normalisation:** Apply NFC Unicode normalisation, standardise curly quotation marks and dashes, and collapse repeated whitespace.
- **Unsupported characters:** Map `à` and `â` to `a`, `è` to `e`, and `ü` to `u` to avoid unknown tokens. Retain the original text fields for traceability. These substitutions improve token compatibility but do not guarantee accurate pronunciation of foreign words or names.
- **Case and punctuation:** Preserve case and punctuation except for the explicitly documented substitutions.
- **Tokenizer:** Use `SpeechT5Tokenizer` from `microsoft/speecht5_tts`, without truncation or padding during inspection. No separate phonemizer is used.
- **Sanity checks:** Check for empty texts, verify repeated normalisation leaves the result unchanged, report sequence lengths, and assert that no unknown tokens remain.
- **Consistency:** Use the same normalisation function for training and inference in both experiments.

Reference: https://huggingface.co/docs/transformers/en/tasks/text-to-speech

## 1.1.4 Train/validation/test split (reproducible)

- **Subset:** Use 3,000 utterances to keep training and experimentation manageable within available compute and time.
- **Split ratio:** 80% training, 10% validation, and 10% test.
- **Seed:** 42.
- **Selection method:** Sort all records by utterance ID, shuffle using a local seeded random generator, select the first 3,000 records, and divide them by position into the three splits.
- **Split sizes:** 2,400 training, 300 validation, and 300 test utterances.
- **Sanity checks:** Verify that all splits are non-empty, their utterance IDs are mutually disjoint, and their combined size equals the selected subset size.
- **Saved protocol:** Store sample assignments in `splits/split_manifest.csv` and configuration details in `splits/split_config.json`. Both experiments must use the same manifest.
- **Validation and test use:** Use validation data for model selection and reserve test data for final evaluation.

In [13]:

# (Optional) Put small sanity-check code here:
# - verify LJSpeech files exist
# - read metadata.csv
# - build split lists
# - print counts

#1.1.1
from pathlib import Path
import csv
from collections import Counter

# Paths relative to the notebook's working directory
DATA_ROOT = Path(os.environ.get("LJSPEECH_ROOT", "LJSpeech-1.1"))
METADATA_PATH = DATA_ROOT / "metadata.csv"
WAV_DIR = DATA_ROOT / "wavs"

assert METADATA_PATH.is_file(), (
    f"Metadata not found: {METADATA_PATH.resolve()}\n"
    f"Current working directory: {Path.cwd()}"
)
assert WAV_DIR.is_dir(), f"Audio folder not found: {WAV_DIR.resolve()}"

# LJSpeech metadata: ID | original text | normalised text
records = []

with METADATA_PATH.open(encoding="utf-8", newline="") as f:
    reader = csv.reader(f, delimiter="|", quoting=csv.QUOTE_NONE)

    for line_number, row in enumerate(reader, start=1):
        if len(row) != 3:
            raise ValueError(
                f"Invalid metadata at line {line_number}: "
                f"expected 3 fields, found {len(row)}"
            )

        utterance_id, raw_text, normalised_text = row

        records.append({
            "utterance_id": utterance_id.strip(),
            "raw_text": raw_text.strip(),
            "text": normalised_text.strip(),
            "audio_path": WAV_DIR / f"{utterance_id.strip()}.wav",
        })

assert records, "Metadata is empty."

# Dataset sanity checks
id_counts = Counter(r["utterance_id"] for r in records)
duplicate_ids = [uid for uid, count in id_counts.items() if count > 1]
empty_ids = [i for i, r in enumerate(records) if not r["utterance_id"]]
empty_text_ids = [r["utterance_id"] for r in records if not r["text"]]
missing_audio = [
    r["utterance_id"] for r in records if not r["audio_path"].is_file()
]

print(f"Metadata samples: {len(records):,}")
print(f"WAV files in folder: {sum(1 for _ in WAV_DIR.glob('*.wav')):,}")
print(f"Duplicate IDs: {len(duplicate_ids)}")
print(f"Empty IDs: {len(empty_ids)}")
print(f"Empty normalised texts: {len(empty_text_ids)}")
print(f"Missing audio files: {len(missing_audio)}")

assert not duplicate_ids, f"Duplicate IDs: {duplicate_ids[:5]}"
assert not empty_ids, f"Empty IDs at record indices: {empty_ids[:5]}"
assert not empty_text_ids, f"Empty texts: {empty_text_ids[:5]}"
assert not missing_audio, f"Missing audio: {missing_audio[:5]}"

print("\nFirst 3 samples:")
for record in records[:3]:
    print(f"\nID: {record['utterance_id']}")
    print(f"Original: {record['raw_text']}")
    print(f"Normalised: {record['text']}")
    print(f"Audio: {record['audio_path']}")

print("\nMetadata and audio-path checks passed.")

Metadata samples: 13,100
WAV files in folder: 13,100
Duplicate IDs: 0
Empty IDs: 0
Empty normalised texts: 0
Missing audio files: 0

First 3 samples:

ID: LJ001-0001
Original: Printing, in the only sense with which we are at present concerned, differs from most if not from all the arts and crafts represented in the Exhibition
Normalised: Printing, in the only sense with which we are at present concerned, differs from most if not from all the arts and crafts represented in the Exhibition
Audio: E:\work\datasets\LJSpeech-1.1\wavs\LJ001-0001.wav

ID: LJ001-0002
Original: in being comparatively modern.
Normalised: in being comparatively modern.
Audio: E:\work\datasets\LJSpeech-1.1\wavs\LJ001-0002.wav

ID: LJ001-0003
Original: For although the Chinese took impressions from wood blocks engraved in relief for centuries before the woodcutters of the Netherlands, by a similar process
Normalised: For although the Chinese took impressions from wood blocks engraved in relief for centuries before t

In [14]:
# 1.1.2 Inspect audio, resample to 16 kHz, and verify examples.
# Audio dependencies: numpy, scipy, soundfile

import math
import wave
import statistics
from collections import Counter

import numpy as np
import soundfile as sf
from scipy.signal import resample_poly
from IPython.display import Audio, display

TARGET_SAMPLE_RATE = 16000

# Inspect all original WAV headers
audio_stats = []
audio_errors = []

for record in records:
    try:
        with wave.open(str(record["audio_path"]), "rb") as wav:
            sample_rate = wav.getframerate()
            channels = wav.getnchannels()
            frames = wav.getnframes()

        if sample_rate <= 0 or frames <= 0:
            raise ValueError("Invalid sample rate or empty audio")

        audio_stats.append({
            "utterance_id": record["utterance_id"],
            "sample_rate": sample_rate,
            "channels": channels,
            "duration": frames / sample_rate,
        })

    except (wave.Error, OSError, ValueError, EOFError) as error:
        audio_errors.append((record["utterance_id"], str(error)))

print(f"Audio headers checked: {len(records):,}")
print(f"Valid headers: {len(audio_stats):,}")
print(f"Audio errors: {len(audio_errors)}")

if audio_errors:
    print("First errors:", audio_errors[:5])

assert audio_stats, "No valid audio headers found."
assert not audio_errors, "Resolve audio errors before continuing."

durations = [item["duration"] for item in audio_stats]

print("\nSample rates:", dict(Counter(
    item["sample_rate"] for item in audio_stats
)))
print("Channel counts:", dict(Counter(
    item["channels"] for item in audio_stats
)))
print(f"Shortest audio: {min(durations):.2f} seconds")
print(f"Longest audio: {max(durations):.2f} seconds")
print(f"Mean duration: {statistics.mean(durations):.2f} seconds")
print(f"Median duration: {statistics.median(durations):.2f} seconds")
print(f"Total duration: {sum(durations) / 3600:.2f} hours")


# Load and resample on demand; preserve original files
def load_audio_for_speecht5(audio_path):
    waveform, original_sr = sf.read(
        str(audio_path),
        dtype="float32",
        always_2d=True,
    )

    if waveform.size == 0 or not np.isfinite(waveform).all():
        raise ValueError(f"Empty or non-finite audio: {audio_path}")

    waveform = waveform.mean(axis=1)

    if original_sr != TARGET_SAMPLE_RATE:
        divisor = math.gcd(original_sr, TARGET_SAMPLE_RATE)
        waveform = resample_poly(
            waveform,
            up=TARGET_SAMPLE_RATE // divisor,
            down=original_sr // divisor,
        )

    waveform = np.asarray(waveform, dtype=np.float32)

    if waveform.size == 0 or not np.isfinite(waveform).all():
        raise ValueError(f"Invalid processed audio: {audio_path}")

    return waveform, TARGET_SAMPLE_RATE


# Verify and play three processed text–audio pairs
example_indices = sorted(set([
    0,
    len(records) // 2,
    len(records) - 1,
]))

for index in example_indices:
    record = records[index]
    waveform, sample_rate = load_audio_for_speecht5(
        record["audio_path"]
    )

    original_info = sf.info(str(record["audio_path"]))
    processed_duration = len(waveform) / sample_rate

    assert sample_rate == TARGET_SAMPLE_RATE
    assert waveform.ndim == 1
    assert waveform.dtype == np.float32
    assert abs(
        processed_duration - original_info.duration
    ) <= 1 / sample_rate

    print(f"\nSample ID: {record['utterance_id']}")
    print(f"Text: {record['text']}")
    print(
        f"Sample rate: {original_info.samplerate} "
        f"-> {sample_rate} Hz"
    )
    print(f"Shape: {waveform.shape}; dtype: {waveform.dtype}")
    print(
        f"Duration: {original_info.duration:.4f} "
        f"-> {processed_duration:.4f} seconds"
    )

    display(Audio(waveform, rate=sample_rate))

print("\nAudio preprocessing checks passed.")

Audio headers checked: 13,100
Valid headers: 13,100
Audio errors: 0

Sample rates: {22050: 13100}
Channel counts: {1: 13100}
Shortest audio: 1.11 seconds
Longest audio: 10.10 seconds
Mean duration: 6.57 seconds
Median duration: 6.76 seconds
Total duration: 23.92 hours

Sample ID: LJ001-0001
Text: Printing, in the only sense with which we are at present concerned, differs from most if not from all the arts and crafts represented in the Exhibition
Sample rate: 22050 -> 16000 Hz
Shape: (154481,); dtype: float32
Duration: 9.6550 -> 9.6551 seconds



Sample ID: LJ023-0097
Text: Justice Stone said of the majority opinion that it was a "tortured construction of the Constitution."
Sample rate: 22050 -> 16000 Hz
Shape: (114357,); dtype: float32
Duration: 7.1473 -> 7.1473 seconds



Sample ID: LJ050-0278
Text: the recommendations we have here suggested would greatly advance the security of the office without any impairment of our fundamental liberties.
Sample rate: 22050 -> 16000 Hz
Shape: (142778,); dtype: float32
Duration: 8.9236 -> 8.9236 seconds



Audio preprocessing checks passed.


In [15]:
# 1.1.3 Normalise text and verify SpeechT5 token compatibility.
# Reference: https://huggingface.co/docs/transformers/en/tasks/text-to-speech
# Tokenizer dependencies: transformers, sentencepiece

import re
import unicodedata
from transformers import SpeechT5Tokenizer

CHECKPOINT = "microsoft/speecht5_tts"
tokenizer = SpeechT5Tokenizer.from_pretrained(CHECKPOINT)


def normalise_text(text):
    text = unicodedata.normalize("NFC", text)

    replacements = {
        "\u2018": "'",
        "\u2019": "'",
        "\u201c": '"',
        "\u201d": '"',
        "\u2013": "-",
        "\u2014": "-",
        "à": "a",
        "â": "a",
        "è": "e",
        "ü": "u",
    }

    for original, replacement in replacements.items():
        text = text.replace(original, replacement)

    return re.sub(r"\s+", " ", text).strip()


changed_records = []
unknown_examples = []
token_lengths = []
total_unknown_tokens = 0

for record in records:
    clean_text = normalise_text(record["text"])

    assert clean_text, f"Empty text: {record['utterance_id']}"
    assert normalise_text(clean_text) == clean_text

    record["clean_text"] = clean_text

    if clean_text != record["text"]:
        changed_records.append(record)

    input_ids = tokenizer(
        clean_text,
        truncation=False,
        padding=False,
    )["input_ids"]

    token_lengths.append(len(input_ids))
    unknown_count = input_ids.count(tokenizer.unk_token_id)
    total_unknown_tokens += unknown_count

    if unknown_count:
        unknown_examples.append(record["utterance_id"])

print(f"Tokenizer: {CHECKPOINT}")
print(f"Texts processed: {len(records):,}")
print(f"Texts changed: {len(changed_records):,}")
print(f"Shortest token sequence: {min(token_lengths)}")
print(f"Longest token sequence: {max(token_lengths)}")
print(f"Texts containing <unk>: {len(unknown_examples)}")
print(f"Total <unk> tokens: {total_unknown_tokens}")

print("\nCleaning examples:")
for record in changed_records[:3]:
    print(f"\nID: {record['utterance_id']}")
    print("Before:", record["text"])
    print("After: ", record["clean_text"])

example_text = records[0]["clean_text"]
example_ids = tokenizer(example_text)["input_ids"]

print("\nTokenisation example:")
print("Text:", example_text)
print("First 30 tokens:", tokenizer.convert_ids_to_tokens(example_ids[:30]))
print("First 30 IDs:", example_ids[:30])

assert not unknown_examples, (
    f"Unsupported text remains: {unknown_examples[:5]}"
)

print("\nText preprocessing and token compatibility checks passed.")

Tokenizer: microsoft/speecht5_tts
Texts processed: 13,100
Texts changed: 22
Shortest token sequence: 14
Longest token sequence: 189
Texts containing <unk>: 0
Total <unk> tokens: 0

Cleaning examples:

ID: LJ011-0058
Before: He was summoned to the Mansion House, where he repeated his request, crying, "Accordez moi cette grâce," with much urgency.
After:  He was summoned to the Mansion House, where he repeated his request, crying, "Accordez moi cette grace," with much urgency.

ID: LJ015-0146
Before: He had a nice taste in bric-à-brac, and was considered a good judge of pictures.
After:  He had a nice taste in bric-a-brac, and was considered a good judge of pictures.

ID: LJ016-0273
Before: At Müller's execution there was great competition for front seats,
After:  At Muller's execution there was great competition for front seats,

Tokenisation example:
Text: Printing, in the only sense with which we are at present concerned, differs from most if not from all the arts and crafts represent

In [16]:
#1.1.4

import random
import json
import csv
from pathlib import Path

# Shared dataset configuration for both experiments
SPLIT_SEED = 42
SUBSET_SIZE = 3000
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9
assert 3 <= SUBSET_SIZE <= len(records)

# Sort first so selection does not depend on metadata row order
shuffled_records = sorted(
    records,
    key=lambda record: record["utterance_id"]
)
random.Random(SPLIT_SEED).shuffle(shuffled_records)

# Select the subset and split it without changing the original records
subset_records = shuffled_records[:SUBSET_SIZE]

n_train = int(SUBSET_SIZE * TRAIN_RATIO)
n_val = int(SUBSET_SIZE * VAL_RATIO)

train_records = subset_records[:n_train]
val_records = subset_records[n_train:n_train + n_val]
test_records = subset_records[n_train + n_val:]

splits = {
    "train": train_records,
    "validation": val_records,
    "test": test_records,
}

# Check that no utterance ID appears in multiple splits
split_ids = {
    name: {record["utterance_id"] for record in items}
    for name, items in splits.items()
}

assert all(len(items) > 0 for items in splits.values())
assert split_ids["train"].isdisjoint(split_ids["validation"])
assert split_ids["train"].isdisjoint(split_ids["test"])
assert split_ids["validation"].isdisjoint(split_ids["test"])

all_ids = set().union(*split_ids.values())
assert len(all_ids) == SUBSET_SIZE

# Save portable paths relative to DATA_ROOT
output_dir = Path("splits")
output_dir.mkdir(parents=True, exist_ok=True)

manifest_path = output_dir / "split_manifest.csv"

with manifest_path.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "utterance_id",
            "split",
            "audio_relative_path",
            "raw_text",
            "text",
            "clean_text",
        ],
    )
    writer.writeheader()

    for split_name, items in splits.items():
        for record in items:
            writer.writerow({
                "utterance_id": record["utterance_id"],
                "split": split_name,
                "audio_relative_path": (
                    record["audio_path"].relative_to(DATA_ROOT).as_posix()
                ),
                "raw_text": record["raw_text"],
                "text": record["text"],
                "clean_text": record["clean_text"],
            })

config = {
    "seed": SPLIT_SEED,
    "subset_size": SUBSET_SIZE,
    "ratios": {
        "train": TRAIN_RATIO,
        "validation": VAL_RATIO,
        "test": TEST_RATIO,
    },
    "counts": {
        name: len(items) for name, items in splits.items()
    },
    "selection_method": (
        "Sort by utterance ID, shuffle with Python random.Random(seed), "
        "select the first subset_size records, then split by position."
    ),
}

config_path = output_dir / "split_config.json"
with config_path.open("w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print(f"Subset size: {SUBSET_SIZE:,}")
print(f"Seed: {SPLIT_SEED}")

for name, items in splits.items():
    print(
        f"{name.capitalize():12s}: {len(items):,} "
        f"({len(items) / SUBSET_SIZE:.0%})"
    )

print("\nUtterance ID overlap: 0")
print(f"Manifest saved to: {manifest_path}")
print(f"Configuration saved to: {config_path}")

Subset size: 3,000
Seed: 42
Train       : 2,400 (80%)
Validation  : 300 (10%)
Test        : 300 (10%)

Utterance ID overlap: 0
Manifest saved to: splits\split_manifest.csv
Configuration saved to: splits\split_config.json


# 1.2 End-to-end TTS pipeline implementation

## 1.2.1 Pipeline overview

We fine-tune `microsoft/speecht5_tts` on the committed LJSpeech subset.
Text uses A's `normalise_text` rules and the SpeechT5 tokenizer; audio uses A's
mono float32 loading and 16 kHz polyphase resampling. SpeechT5 predicts 80-bin
log-mel acoustic features. The pretrained `microsoft/speecht5_hifigan` vocoder
converts these features into a mono 16 kHz waveform saved as PCM16 WAV.
The vocoder is fixed during our acoustic-model fine-tuning.

`Input text -> A's text normalization -> SpeechT5 tokenizer -> SpeechT5 acoustic model -> fixed HiFi-GAN -> WAV`

SpeechT5 also receives a 512-dimensional speaker vector. For formal runs, we use a
fixed vector obtained by averaging normalized x-vectors from 16 training recordings,
then normalizing the mean. The same vector conditions all training/validation items
and final inference for this single-speaker task. Reference IDs, audio hashes and
manifest hash are saved, and no validation/test audio is used to construct this vector.
The debug run uses a public CMU ARCTIC vector only to check the software path.

The pretrained starting point and small LJSpeech subset make experimentation feasible
on the verified local RTX 5070 Ti (approximately 16 GB VRAM). We start with batch size 1,
float32 precision and non-reentrant gradient checkpointing; this configuration still
requires a local integration run to establish actual memory usage and runtime.

## 1.2.2 Environment and dependencies

Verified during the earlier standalone local checks: Windows, Python 3.11.15,
PyTorch/torchaudio 2.8.0+cu128, Transformers 4.57.6, and RTX 5070 Ti.
The displayed CUDA runtime is 12.8. `requirements.txt` defines the shared environment;
`environment.json` below records the environment actually running this notebook.
A's existing checks need NumPy, SciPy, SoundFile, SentencePiece and IPython.
SpeechBrain 1.0.3 is additionally required for the formal training-only speaker vector.

## 1.2.3 Shared training and model selection

The source cells below write the same modules supplied in this repository, so the
implementation is included in the notebook as well as available from the command line.
`tts_b.py` reads `splits/split_manifest.csv` without regenerating assignments and uses
its `clean_text` and `audio_relative_path` columns. Each run records SHA256 checksums,
actual train/validation IDs, source revision, software/hardware and all active settings.
Test records are held out. Validation uses a fixed temporary RNG state for the
SpeechT5 prenet's evaluation-time dropout, restoring the training RNG afterwards.
The best checkpoint minimizes the utterance-weighted mean validation batch loss,
with the first checkpoint winning ties. Log entries distinguish train and validation loss.

Gradient accumulation controls the effective batch size. The example formal configs
use 1 x 4 = 4 utterances per optimizer step, AdamW, constant learning rate, weight decay
0.01 and gradient clipping at 1.0. Model checkpoints contain inference weights and
processor/speaker files; optimizer-state resume is not implemented.
The first integrated run writes `configs/base_model.lock.json`; commit that lock before
both experiments so they start from the same pinned pretrained model revision.

Commands, from the repository root:

```text
python tts_b.py check --data-root <LJSPEECH_ROOT>
python tts_b.py train --config configs/smoke.json --data-root <LJSPEECH_ROOT>
python tts_b.py speaker --data-root <LJSPEECH_ROOT>
python tts_b.py train --config configs/expA.example.json --data-root <LJSPEECH_ROOT>
python tts_b.py train --config configs/expB.example.json --data-root <LJSPEECH_ROOT>
```

The expA/expB configurations are proposed examples, pending the group's decision.
No formal training or evaluation result is claimed by this integration.

## 1.2.4 Checkpoint interface for inference

B supplies a small checkpoint-reload helper for integration testing; C can reuse or
extend it for the final synthesis/evaluation implementation. Each checkpoint carries
its matching `speaker.npy` and `speaker.json` alongside model and processor files.

```text
python tts_b.py synth --checkpoint <RUN_DIRECTORY>/best --text "Hello world." --out wav_examples/demo.wav
```

The actual checkpoint path is printed and saved in `results/latest_<experiment>.json`.
Sources: https://huggingface.co/microsoft/speecht5_tts ;
https://huggingface.co/microsoft/speecht5_hifigan ;
https://huggingface.co/docs/transformers/tasks/text-to-speech ;
https://huggingface.co/speechbrain/spkrec-xvect-voxceleb .


In [17]:
%%writefile tts_protocol.py
"""Shared protocol checks. Text processing is copied from A's notebook."""
import csv
import hashlib
import json
from pathlib import Path, PurePosixPath
import re
import unicodedata
from collections import Counter


def normalise_text(text):
    text = unicodedata.normalize("NFC", text)
    replacements = {
        "\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"',
        "\u2013": "-", "\u2014": "-", "à": "a", "â": "a", "è": "e", "ü": "u",
    }
    for original, replacement in replacements.items():
        text = text.replace(original, replacement)
    return re.sub(r"\s+", " ", text).strip()


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def read_protocol(manifest, split_config, data_root, check_files=True):
    """Read the committed manifest; never create or rewrite a split."""
    manifest, split_config, data_root = map(Path, (manifest, split_config, data_root))
    config = json.loads(split_config.read_text(encoding="utf-8-sig"))
    with manifest.open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        required = {"utterance_id", "split", "audio_relative_path", "text", "clean_text"}
        if not required.issubset(reader.fieldnames or []):
            raise ValueError(f"Manifest needs columns {sorted(required)}")
        rows = list(reader)
    groups = {name: [] for name in ("train", "validation", "test")}
    seen = set()
    for row in rows:
        uid = row["utterance_id"]
        if not uid or uid in seen:
            raise ValueError(f"Empty or duplicate utterance ID: {uid!r}")
        seen.add(uid)
        if row["split"] not in groups:
            raise ValueError(f"Unknown split for {uid}: {row['split']}")
        relative = PurePosixPath(row["audio_relative_path"])
        if (relative.is_absolute() or relative.parts != ("wavs", f"{uid}.wav")
                or "\\" in row["audio_relative_path"] or ":" in row["audio_relative_path"]):
            raise ValueError(f"Unexpected audio path for {uid}: {relative}")
        if not row["clean_text"] or normalise_text(row["text"]) != row["clean_text"]:
            raise ValueError(f"clean_text differs from A's normalization for {uid}")
        item = dict(row)
        item["audio_path"] = data_root.joinpath(*relative.parts)
        if check_files and not item["audio_path"].is_file():
            raise FileNotFoundError(item["audio_path"])
        groups[row["split"]].append(item)
    counts = {name: len(items) for name, items in groups.items()}
    if counts != config["counts"] or sum(counts.values()) != config["subset_size"]:
        raise ValueError(f"Counts {counts} differ from split_config.json")
    if any(count == 0 for count in counts.values()):
        raise ValueError("An empty split was found")
    return groups, {
        "manifest_sha256": sha256_file(manifest),
        # Canonical row content is stable across Git's LF/CRLF conversion.
        # Row order is retained because it also affects seeded sampling.
        "manifest_content_sha256": hashlib.sha256(json.dumps(
            rows, ensure_ascii=False, sort_keys=True, separators=(",", ":")
        ).encode("utf-8")).hexdigest(),
        "split_config_sha256": sha256_file(split_config),
        "counts": counts, "seed": config["seed"],
    }


Overwriting tts_protocol.py


In [18]:
%%writefile tts_audio.py
"""A's loading/resampling rule, reused by B's training and speaker extraction."""
import math
import numpy as np
import soundfile as sf
from scipy.signal import resample_poly

TARGET_SAMPLE_RATE = 16000


def load_audio_for_speecht5(audio_path):
    waveform, original_sr = sf.read(str(audio_path), dtype="float32", always_2d=True)
    if waveform.size == 0 or not np.isfinite(waveform).all():
        raise ValueError(f"Empty or non-finite audio: {audio_path}")
    waveform = waveform.mean(axis=1)
    if original_sr != TARGET_SAMPLE_RATE:
        divisor = math.gcd(original_sr, TARGET_SAMPLE_RATE)
        waveform = resample_poly(waveform, up=TARGET_SAMPLE_RATE // divisor,
                                 down=original_sr // divisor)
    waveform = np.asarray(waveform, dtype=np.float32)
    if waveform.size == 0 or not np.isfinite(waveform).all():
        raise ValueError(f"Invalid processed audio: {audio_path}")
    return waveform, TARGET_SAMPLE_RATE


Overwriting tts_audio.py


In [19]:
%%writefile tts_b.py
"""B's shared SpeechT5 training entry point; runtime imports are lazy.

Sources: Hugging Face's SpeechT5 TTS guide, model cards for
microsoft/speecht5_tts and microsoft/speecht5_hifigan, and SpeechBrain's
speechbrain/spkrec-xvect-voxceleb model card. A's processing is in tts_protocol
and tts_audio. Experiment results are produced only by actual local runs.
"""
import argparse
from datetime import datetime, timezone
import gc
import io
import json
import math
import os
from pathlib import Path
import platform
import random
import subprocess
import sys
import zipfile
from tts_protocol import normalise_text, read_protocol, sha256_file

ROOT = Path(__file__).resolve().parent
MODEL_ID = "microsoft/speecht5_tts"
VOCODER_ID = "microsoft/speecht5_hifigan"
PUBLIC_SPEAKER_ID = "Matthijs/cmu-arctic-xvectors"


def cache_path():
    default = (r"E:\work\_cache\huggingface" if os.name == "nt" and Path("E:/work").is_dir()
               else str(Path.home() / ".cache/huggingface"))
    root = Path(os.environ.get("TTS_HF_CACHE", os.environ.get("HF_HOME", default)))
    os.environ["HF_HOME"] = str(root)
    (root / "hub").mkdir(parents=True, exist_ok=True)
    return root


def runtime():
    cache_path()  # before importing the Hub
    global np, sf, torch, transformers
    global SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan
    import numpy as np
    import soundfile as sf
    import torch
    import transformers
    from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan


def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, ensure_ascii=False), encoding="utf-8")


def locked_base_model():
    """The first successful load pins the shared base revision for both runs."""
    lock_path = ROOT / "configs/base_model.lock.json"
    lock = json.loads(lock_path.read_text()) if lock_path.exists() else {}
    if lock and (lock.get("model") != MODEL_ID or not lock.get("revision")):
        raise ValueError("Invalid shared base-model lock")
    kwargs = {"cache_dir": str(cache_path() / "hub")}
    if lock:
        kwargs["revision"] = lock["revision"]
    model = SpeechT5ForTextToSpeech.from_pretrained(MODEL_ID, **kwargs)
    revision = getattr(model.config, "_commit_hash", None)
    if not revision:
        raise RuntimeError("Could not identify the base-model revision for reproducibility")
    processor = SpeechT5Processor.from_pretrained(MODEL_ID, revision=revision,
                                                 cache_dir=str(cache_path() / "hub"))
    if not lock:
        write_json(lock_path, {"model": MODEL_ID, "revision": revision})
    return model, processor


def protocol(args, check_files=True):
    return read_protocol(ROOT / "splits/split_manifest.csv",
                         ROOT / "splits/split_config.json", args.data_root, check_files)


def check(args):
    groups, info = protocol(args, check_files=not args.manifest_only)
    print(json.dumps(info, indent=2))
    print("Audio paths checked:", not args.manifest_only)
    print("MANIFEST CHECK PASSED; committed split preserved.")


def prepare_speaker(args):
    """Create one fixed LJ voice vector using only training references."""
    runtime()
    from tts_audio import load_audio_for_speecht5
    from speechbrain.inference.classifiers import EncoderClassifier
    from speechbrain.utils.fetching import LocalStrategy
    groups, info = protocol(args)
    if not 1 <= args.references <= len(groups["train"]):
        raise ValueError("Invalid training-reference count")
    refs = sorted(groups["train"], key=lambda item: item["utterance_id"])[:args.references]
    device = "cuda" if torch.cuda.is_available() else "cpu"
    encoder = EncoderClassifier.from_hparams(
        source="speechbrain/spkrec-xvect-voxceleb",
        savedir=str(cache_path() / "speaker_encoder"),
        huggingface_cache_dir=str(cache_path() / "hub"),
        local_strategy=LocalStrategy.COPY, run_opts={"device": device},
    )
    vectors, reference_hashes = [], {}
    with torch.inference_mode():
        for index, row in enumerate(refs, 1):
            audio, _ = load_audio_for_speecht5(row["audio_path"])
            v = encoder.encode_batch(torch.from_numpy(audio).unsqueeze(0).to(device)).reshape(-1)
            if v.numel() != 512 or not torch.isfinite(v).all() or v.norm().item() == 0:
                raise ValueError(f"Invalid x-vector: {row['utterance_id']}")
            vectors.append(torch.nn.functional.normalize(v, dim=0).cpu())
            reference_hashes[row["utterance_id"]] = sha256_file(row["audio_path"])
            print(f"Training speaker reference {index}/{len(refs)}: {row['utterance_id']}", flush=True)
    vector = torch.stack(vectors).mean(0)
    vector = torch.nn.functional.normalize(vector, dim=0).numpy().astype(np.float32)
    out = Path(args.out).resolve()
    if out.suffix != ".npy":
        raise ValueError("--out must end in .npy")
    out.parent.mkdir(parents=True, exist_ok=True)
    if out.exists() or out.with_suffix(".json").exists():
        raise FileExistsError(f"Speaker files already exist: {out}. Choose another --out to regenerate.")
    np.save(out, vector, allow_pickle=False)
    write_json(out.with_suffix(".json"), {
        "method": "L2-normalized mean of L2-normalized x-vectors from training references",
        "encoder": "speechbrain/spkrec-xvect-voxceleb", "reference_split": "train",
        "reference_ids": list(reference_hashes), "reference_wav_sha256": reference_hashes,
        "manifest_sha256": info["manifest_sha256"], "vector_sha256": sha256_file(out),
        "manifest_content_sha256": info["manifest_content_sha256"],
        "selection": "First N training IDs in sorted order", "references": len(refs),
    })
    print("Training-only speaker vector saved:", out)


def speaker_vector(args, groups, info, debug):
    if debug:
        from huggingface_hub import hf_hub_download
        archive = hf_hub_download(repo_id=PUBLIC_SPEAKER_ID, repo_type="dataset",
                                  filename="spkrec-xvect.zip", cache_dir=str(cache_path() / "hub"))
        with zipfile.ZipFile(archive) as handle:
            names = sorted(n for n in handle.namelist() if n.endswith(".npy"))
            preferred = [n for n in names if "cmu_us_slt_arctic" in n]
            if not names:
                raise ValueError("No public debug speaker vectors found")
            member = (preferred or names)[0]
            vector = np.load(io.BytesIO(handle.read(member)), allow_pickle=False)
        metadata = {"debug_only": True, "source": PUBLIC_SPEAKER_ID, "member": member}
    else:
        path = Path(args.speaker)
        metadata = json.loads(path.with_suffix(".json").read_text(encoding="utf-8-sig"))
        vector = np.load(path, allow_pickle=False)
        references = metadata.get("reference_ids", [])
        train_ids = {row["utterance_id"] for row in groups["train"]}
        if (metadata.get("reference_split") != "train" or not references
                or not set(references).issubset(train_ids)
                or metadata.get("manifest_content_sha256") != info["manifest_content_sha256"]
                or metadata.get("vector_sha256") != sha256_file(path)):
            raise ValueError("Speaker provenance does not match this training split")
    vector = np.asarray(vector, dtype=np.float32).reshape(-1)
    if vector.shape != (512,) or not np.isfinite(vector).all() or np.linalg.norm(vector) == 0:
        raise ValueError("Invalid speaker vector")
    vector = vector / np.linalg.norm(vector)
    return vector, metadata


def features(rows, processor, factor):
    from tts_audio import load_audio_for_speecht5
    result = []
    for index, row in enumerate(rows, 1):
        audio, rate = load_audio_for_speecht5(row["audio_path"])
        item = processor(text=row["clean_text"], audio_target=audio,
                         sampling_rate=rate, return_attention_mask=False)
        ids = item["input_ids"]
        if len(ids) > 450 or processor.tokenizer.unk_token_id in ids:
            raise ValueError(f"Unsupported/overlong input for {row['utterance_id']}; no rows are silently dropped")
        labels = np.asarray(item["labels"][0], dtype=np.float32)
        length = len(labels) - len(labels) % factor
        if labels.ndim != 2 or labels.shape[1] != 80 or length < factor or not np.isfinite(labels).all():
            raise ValueError(f"Invalid labels for {row['utterance_id']}")
        result.append({"input_ids": ids, "labels": labels[:length]})
        if index % 100 == 0 or index == len(rows):
            print(f"Prepared {index}/{len(rows)} utterances", flush=True)
    return result


def batch(items, processor, vector, device):
    value = processor.pad(
        input_ids=[{"input_ids": item["input_ids"]} for item in items],
        labels=[{"input_values": item["labels"]} for item in items], return_tensors="pt",
    )
    value["labels"] = value["labels"].masked_fill(value["decoder_attention_mask"].unsqueeze(-1).ne(1), -100)
    del value["decoder_attention_mask"]
    value["speaker_embeddings"] = vector.expand(len(items), -1)
    return {key: tensor.to(device) for key, tensor in value.items()}


def validation(model, items, processor, vector, device, batch_size, seed):
    model.eval()
    total = 0.0
    # SpeechT5's decoder prenet uses dropout during evaluation. Fix and restore
    # RNG state so model selection is comparable and does not change training RNG.
    with torch.random.fork_rng(devices=[device.index or 0]):
        torch.manual_seed(seed + 100000)
        with torch.no_grad():
            for start in range(0, len(items), batch_size):
                part = items[start:start + batch_size]
                loss = float(model(**batch(part, processor, vector, device)).loss.cpu())
                if not math.isfinite(loss):
                    raise RuntimeError("Non-finite validation loss")
                total += loss * len(part)
    model.train()
    return total / len(items)


def train(args):
    runtime()
    if not torch.cuda.is_available():
        raise RuntimeError("Training requires the verified CUDA environment")
    cfg = json.loads(Path(args.config).read_text(encoding="utf-8-sig"))
    groups, info = protocol(args)
    debug = bool(cfg.get("debug", False))
    for key in ("max_steps", "batch_size", "gradient_accumulation_steps", "eval_steps"):
        if not isinstance(cfg.get(key), int) or cfg[key] <= 0:
            raise ValueError(f"Invalid config value: {key}")
    if not math.isfinite(cfg["learning_rate"]) or cfg["learning_rate"] <= 0:
        raise ValueError("Invalid learning rate")
    if cfg["seed"] != info["seed"]:
        raise ValueError("Training seed differs from the committed split protocol")
    random.seed(cfg["seed"])
    np.random.seed(cfg["seed"])
    torch.manual_seed(cfg["seed"])
    torch.cuda.manual_seed_all(cfg["seed"])
    torch.backends.cudnn.benchmark = False
    device = torch.device("cuda", torch.cuda.current_device())
    vector_np, speaker_info = speaker_vector(args, groups, info, debug)
    vector = torch.from_numpy(vector_np).unsqueeze(0).to(device)
    run = ROOT / "runs" / (cfg["name"] + "_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
    run.mkdir(parents=True, exist_ok=False)
    train_rows = groups["train"][:16] if debug else groups["train"]
    val_rows = groups["validation"][:4] if debug else groups["validation"]
    print("Run:", run, flush=True)
    print("DEBUG ONLY" if debug else "FORMAL TRAINING; TEST SET HELD OUT", flush=True)
    print("Counts used:", len(train_rows), "train /", len(val_rows), "validation", flush=True)
    try:
        git_head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True, stderr=subprocess.DEVNULL).strip()
        git_dirty = bool(subprocess.check_output(["git", "status", "--porcelain"], cwd=ROOT, text=True))
    except (OSError, subprocess.CalledProcessError):
        git_head, git_dirty = None, None
    model, processor = locked_base_model()
    model = model.to(device)
    model.config.use_cache = False
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    record = {
        "config": cfg, "protocol": info, "data_root": str(Path(args.data_root).resolve()),
        "train_ids_used": [row["utterance_id"] for row in train_rows],
        "validation_ids_used": [row["utterance_id"] for row in val_rows],
        "speaker": speaker_info, "seed": cfg["seed"], "precision": "float32",
        "optimizer": "AdamW", "weight_decay": 0.01, "lr_schedule": "constant",
        "gradient_checkpointing": {"use_reentrant": False},
        "checkpoint_rule": "lowest validation loss; first checkpoint wins ties",
        "python": sys.version, "platform": platform.platform(), "torch": torch.__version__,
        "transformers": transformers.__version__, "gpu": torch.cuda.get_device_name(device),
        "model": MODEL_ID, "model_revision": getattr(model.config, "_commit_hash", None),
        "git_head": git_head, "git_dirty": git_dirty,
        "started_utc": datetime.now(timezone.utc).isoformat(),
    }
    write_json(run / "run_config.json", record)
    train_items = features(train_rows, processor, model.config.reduction_factor)
    val_items = features(val_rows, processor, model.config.reduction_factor)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["learning_rate"], weight_decay=0.01)
    order, cursor = [], 0
    sampler = random.Random(cfg["seed"])
    best_loss, best_step = float("inf"), None
    model.train()
    for step in range(1, cfg["max_steps"] + 1):
        optimizer.zero_grad(set_to_none=True)
        losses = []
        for _ in range(cfg["gradient_accumulation_steps"]):
            picked = []
            for _ in range(cfg["batch_size"]):
                if cursor == len(order):
                    order = list(range(len(train_items)))
                    sampler.shuffle(order)
                    cursor = 0
                picked.append(train_items[order[cursor]])
                cursor += 1
            loss = model(**batch(picked, processor, vector, device)).loss
            if not torch.isfinite(loss).item():
                raise RuntimeError(f"Non-finite loss at optimizer step {step}")
            losses.append(float(loss.detach().cpu()))
            (loss / cfg["gradient_accumulation_steps"]).backward()
        norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        if not torch.isfinite(norm).item():
            raise RuntimeError(f"Non-finite gradients at step {step}")
        optimizer.step()
        entry = {"step": step, "train_loss": sum(losses) / len(losses)}
        if step % cfg["eval_steps"] == 0 or step == cfg["max_steps"]:
            score = validation(model, val_items, processor, vector, device, cfg["batch_size"], cfg["seed"])
            entry["validation_loss"] = score
            if score < best_loss:
                best_loss, best_step = score, step
                checkpoint = run / "best"
                model.save_pretrained(checkpoint, safe_serialization=True)
                processor.save_pretrained(checkpoint)
                np.save(checkpoint / "speaker.npy", vector_np, allow_pickle=False)
                write_json(checkpoint / "speaker.json", speaker_info)
                write_json(checkpoint / "training_record.json", record)
        with (run / "log.jsonl").open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(entry) + "\n")
        print(json.dumps(entry), flush=True)
    summary = {"debug": debug, "best_validation_loss": best_loss, "best_step": best_step,
               "best_checkpoint": str(run / "best"), "completed_steps": cfg["max_steps"],
               "manifest_sha256": info["manifest_sha256"], "test_evaluation": "not performed"}
    summary["manifest_content_sha256"] = info["manifest_content_sha256"]
    write_json(run / "summary.json", summary)
    write_json(ROOT / "results" / f"latest_{cfg['name']}.json", summary)
    optimizer.zero_grad(set_to_none=True)
    del optimizer, model, loss
    gc.collect()
    torch.cuda.empty_cache()
    if debug:
        demo = argparse.Namespace(checkpoint=run / "best", text="This is our shared training pipeline.",
                                  out=ROOT / "wav_examples" / f"{run.name}.wav")
        synth(demo)
        print("INTEGRATED TRAINING CHECK PASSED", flush=True)
    else:
        print("TRAINING COMPLETE. Send the best checkpoint and paired speaker files to C.", flush=True)
    print(json.dumps(summary, indent=2), flush=True)


def synth(args):
    """Checkpoint reload helper. C can reuse it for uniform evaluation."""
    runtime()
    checkpoint = Path(args.checkpoint)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    processor = SpeechT5Processor.from_pretrained(checkpoint)
    model = SpeechT5ForTextToSpeech.from_pretrained(checkpoint).to(device).eval()
    model.config.use_cache = True
    vocoder = SpeechT5HifiGan.from_pretrained(VOCODER_ID, cache_dir=str(cache_path() / "hub")).to(device).eval()
    vector = np.load(checkpoint / "speaker.npy", allow_pickle=False).astype(np.float32).reshape(1, -1)
    if vector.shape != (1, 512) or not np.isfinite(vector).all():
        raise ValueError("Invalid checkpoint speaker vector")
    text = normalise_text(args.text)
    if not text:
        raise ValueError("Text is empty")
    ids = processor(text=text, return_tensors="pt")["input_ids"]
    if ids.shape[1] > 450 or (ids == processor.tokenizer.unk_token_id).any().item():
        raise ValueError("Unsupported/overlong text. Spell out numbers and use supported English characters.")
    with torch.inference_mode():
        speech = model.generate_speech(ids.to(device), torch.from_numpy(vector).to(device), vocoder=vocoder)
    audio = speech.cpu().numpy().reshape(-1)
    if audio.size == 0 or not np.isfinite(audio).all() or np.max(np.abs(audio)) == 0:
        raise RuntimeError("Invalid or silent generated audio")
    out = Path(args.out)
    out.parent.mkdir(parents=True, exist_ok=True)
    sf.write(out, audio, int(vocoder.config.sampling_rate), subtype="PCM_16")
    print("Saved WAV:", out, flush=True)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    sub = parser.add_subparsers(dest="command", required=True)
    default_root = os.environ.get("LJSPEECH_ROOT", "LJSpeech-1.1")
    p = sub.add_parser("check")
    p.add_argument("--data-root", default=default_root)
    p.add_argument("--manifest-only", action="store_true")
    p.set_defaults(func=check)
    p = sub.add_parser("speaker")
    p.add_argument("--data-root", default=default_root)
    p.add_argument("--references", type=int, default=16)
    p.add_argument("--out", default=str(ROOT / "speaker/lj_speaker.npy"))
    p.set_defaults(func=prepare_speaker)
    p = sub.add_parser("train")
    p.add_argument("--data-root", default=default_root)
    p.add_argument("--config", required=True)
    p.add_argument("--speaker", default=str(ROOT / "speaker/lj_speaker.npy"))
    p.set_defaults(func=train)
    p = sub.add_parser("synth")
    p.add_argument("--checkpoint", required=True)
    p.add_argument("--text", required=True)
    p.add_argument("--out", required=True)
    p.set_defaults(func=synth)
    args = parser.parse_args()
    args.func(args)


if __name__ == "__main__":
    main()


Overwriting tts_b.py


In [23]:
# Run the manifest check; the full 3000-row assignments are preserved.
import subprocess
import json
import importlib.metadata

DATA_ARGUMENT = os.environ["LJSPEECH_ROOT"]
def run_b(*arguments):
    result = subprocess.run(
        [sys.executable, "tts_b.py", *map(str, arguments)],
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    if result.stdout:
        print(result.stdout, flush=True)
    if result.stderr:
        print(result.stderr, flush=True)
    result.check_returncode()

run_b("check", "--data-root", DATA_ARGUMENT)

packages = ["torch", "torchaudio", "transformers", "numpy", "scipy", "soundfile", "sentencepiece", "speechbrain"]
environment = {"python": sys.version, "executable": sys.executable, "packages": {}}
for name in packages:
    try:
        environment["packages"][name] = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        environment["packages"][name] = "not installed"
Path("environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
print(json.dumps(environment, indent=2))

# Opt-in because training writes model weights and takes time. Set True for the
# integration check; it uses 16 train + 4 validation IDs from A's committed split.
RUN_INTEGRATION_SMOKE = True
if RUN_INTEGRATION_SMOKE:
    run_b("train", "--config", "configs/smoke.json", "--data-root", DATA_ARGUMENT)

# Formal preparation/training is explicit and never runs by default.
# run_b("speaker", "--data-root", DATA_ARGUMENT)
# run_b("train", "--config", "configs/expA.example.json", "--data-root", DATA_ARGUMENT)


{
  "manifest_sha256": "a9c52de146c23328250419b6666fb71fbdb71f05db7ed31114b8f61050fa1b0c",
  "manifest_content_sha256": "64a2db28bb18b5aedcc0be012d80d0e0569a774dad554b2e6238068574dcb885",
  "split_config_sha256": "f38cd9613d38872b00ca26ccc3797af3108e72bbe7092e9ada551ec49dae37ca",
  "counts": {
    "train": 2400,
    "validation": 300,
    "test": 300
  },
  "seed": 42
}
Audio paths checked: True
MANIFEST CHECK PASSED; committed split preserved.

{
  "python": "3.11.15 (main, Mar 10 2026, 18:12:25) [MSC v.1944 64 bit (AMD64)]",
  "executable": "e:\\work\\TTS_A3\\.venv\\Scripts\\python.exe",
  "packages": {
    "torch": "2.8.0+cu128",
    "torchaudio": "2.8.0+cu128",
    "transformers": "4.57.6",
    "numpy": "2.4.6",
    "scipy": "1.17.1",
    "soundfile": "0.14.0",
    "sentencepiece": "0.2.2",
    "speechbrain": "1.0.3"
  }
}
Run: E:\work\Groups-26_A3\runs\integration_smoke_20261029_123216_226358
DEBUG ONLY
Counts used: 16 train / 4 validation
Prepared 16/16 utterances
Prepared 4/4 ut

# 1.3 Experimental methodology

## 1.3.1 Controlled experiments

The committed 2400/300/300 split is shared across all formal experiments.
The supplied `configs/expA.example.json` and `configs/expB.example.json` propose
learning rate as the independent variable: 1e-5 versus 2e-5. All other training
settings are identical except the run label. This is a proposal for A/B/C to confirm,
not a record of completed experiments. Both runs start from the pinned pretrained
model, not the earlier debug checkpoint, and use the same training-derived speaker vector.
If the group chooses another variable, edit these configurations before formal runs.

The debug config is not a controlled experiment and must not appear as expA/expB
in the report's results table.

## 1.3.2 Reproducibility records for C to collate

Each completed run writes `run_config.json`, `log.jsonl`, `summary.json` and `best/`
under its unique `runs/` directory. These include the manifest/config hashes,
actual IDs used, seed, base-model revision, effective batch settings, learning rate,
best validation step and environment. `results/latest_<name>.json` points to the
chosen checkpoint. Share the checkpoint's paired speaker files with C.
Actual final experiment settings, objective metrics and qualitative evidence are still pending.


In [21]:

# (Optional) Run/record your experiment commands here (or print them):
# !python scripts/train.py --config configs/expA.yaml
# !python scripts/train.py --config configs/expB.yaml

# (Required) Produce qualitative samples (≥10 prompts) and save wavs
# Example pattern:
# prompts = [...]
# for i, t in enumerate(prompts):
#     out = f"wav_examples/sample_{i:02d}.wav"
#     synthesize_to_wav(text=t, checkpoint="...", out_path=out)

# (Required) Compute objective metric(s) and report metric results clearly in the notebook.# 1.4 Evaluation, coding practice & attribution (Rubric 1.4)

## 1.4.1 Objective metric (minimum: 1) — report for each experiment
Implement and report at least one objective metric for **each** experiment, e.g.:
- **Latency:** average synthesis time per sentence (state number of runs)  
- **Real-Time Factor (RTF):** synthesis_time / audio_duration  
- **Other justified metric:** `<describe and justify>`

Report results in a small table (in this notebook)

## 1.4.2 Qualitative evaluation (required)
Provide **≥10 text prompts** and corresponding **generated `.wav` outputs** (per your task requirements).  
Include:
- The list of prompts you used
- Where the `.wav` files are saved (`wav_examples/…`)
- (Optional) a short comment on perceptual quality differences between experiments

> Spectrograms are optional unless explicitly required by your task sheet. If you include them, add 2–3 representative samples per experiment in `results/spectrograms/`.

## 1.4.3 Coding practice & clarity

A's dataset functions and committed split are retained. Shared modules separate
protocol checks, audio loading, training and checkpoint reload. Paths are configured
with `LJSPEECH_ROOT` and `TTS_HF_CACHE`; the notebook includes full module source.
Runs use fixed seeds, non-reentrant gradient checkpointing, manifest checksums,
training-only speaker references, explicit configs and isolated output folders.
Before submission, rerun from a fresh kernel, confirm formal expA/expB evidence,
and include the referenced modules/configs/split files in the code ZIP.

## 1.4.4 Attribution / external resources
List any:
- External libraries/models/toolkits used
- Pretrained checkpoints used
- Tutorials or code snippets adapted (with links in your report; in this notebook, name them clearly)

## 1.4.5 AI use statement (if any)
State whether you used generative AI tools, and for what (e.g., debugging, refactoring, documentation).  
If used, specify what you verified yourself and what parts were AI-assisted.

# (Required) Produce qualitative samples (≥10 prompts) and save wavs
# Example pattern:
# prompts = [...]
# for i, t in enumerate(prompts):
#     out = f"wav_examples/sample_{i:02d}.wav"
#     synthesize_to_wav(text=t, checkpoint="...", out_path=out)
# (Required) Compute objective metric(s) and report metric results clearly in the notebook.

# Final verification (before submission)

## Restart & run
Restart the kernel and run all cells you want the marker to execute, ensuring:
- No hidden dependencies on your local machine paths
- Commands/paths work from the submission root

## Confirm required artefacts exist
- `wav_examples/` contains generated `.wav` outputs (**≥10 prompts**)  
- bjective metric results for each experiment are clearly reported in the notebook
- dependency file exists (`requirements.txt` / `environment.yml` / etc.)  
- inference entrypoint works (single command that takes text and outputs wav)

## Marker instructions (one paragraph)
Write a short paragraph here telling the marker **exactly** how to:
1) install deps, 2) run training (or load checkpoints), 3) run inference demo, 4) locate results.